# Robustness Checks for the Dimension Classifiers

Two experiments that probe how *stable* the LLM classifier's dimension labels are — complementary to
the inter-annotator agreement analysis (which asks whether the labels are *valid*):

**Experiment A — Relabel stability.** Take a stratified sample of already-classified racialised
reviews, run the *identical* classifier again (same model, same prompt, temperature 0), and compare
new vs. original labels. Even at temperature 0, LLM APIs are not perfectly deterministic (provider-side
batching/numerics), and the original labels were produced weeks ago — so this measures the *rerun
noise floor*: disagreement we must expect for reasons that have nothing to do with the input text.

**Experiment B — Translation-path sensitivity.** The classifiers only ever saw the **English
translations** (`trans_review`) of mostly-German originals. If the labels depend on incidental phrasing
introduced by translation, that is measurement error correlated with language — a serious problem for
an analysis of racialisation across German regions. We probe this by **back-translation**: English →
German → English (producing a *different but meaning-equivalent* phrasing), then classify the
back-translated text and compare.

**Why both are needed to interpret either.** If back-translated texts disagree with the original labels
12% of the time but plain reruns already disagree 8% of the time, the *translation-specific* effect is
only ~4 points. Experiment A is the control for Experiment B — and because the back-translation sample
is a subset of the relabel sample, the comparison is **paired per review**.

| design decision | value | rationale |
|---|---|---|
| population | 7,768 racialised in-scope reviews (canonical dataset) | what the analysis uses |
| Experiment A sample | n = 300, stratified | see stratification below |
| Experiment B sample | n = 150, stratified subset of A | pairs each back-translation with a same-review rerun |
| stratification key | joint (approach × connotation × context) label combination | rare combinations (e.g. multi-label, `coded`) would be missed by simple random sampling — exactly the cases most likely to be unstable |
| allocation | proportional, minimum 3 per stratum (or all if stratum < 3) | proportionality keeps estimates interpretable; the floor guarantees rare strata appear |
| seed | 20260710 | reproducibility |
| classifier settings | identical scripts, model `claude-sonnet-4-6`, temperature 0, batch_size 1 | must match production exactly or we measure the wrong thing |
| intermediate language | German | the dominant original language of the corpus (~93%) |
| translator | same LLM, temperature 0 | ⚠ the original `trans_review` came from a different translator (scraper-side, presumably Google). We are testing sensitivity to *translation-induced paraphrase*, not replicating the original pipeline |
| dimensions | approach, context, connotation, **and theme** | theme needs no human annotation here — model vs model |

Estimated API cost: 4 dims × 300 + 4 dims × 150 relabels ≈ 1,800 calls ≈ **$12–15**, plus ~300
translation calls ≈ $3.

## 1. Stratified sample

We build the stratification key from the canonical labels, allocate proportionally with a floor, and
freeze the result to `samples/relabel_sample_300.parquet`. The cell is **idempotent**: if the sample
file already exists it is loaded, not regenerated — so the notebook can be re-run without invalidating
completed classifier runs.

In [ ]:
import json
import subprocess
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT   = Path('/home/ethel/Desktop/swimming_pool_project/spp_15052026')
RC     = ROOT / 'robustness_check'
CANON  = ROOT / 'analysis_10072026' / 'gmaps_data' / 'racialised_reviews_analysis.parquet'
MASTER = ROOT / 'analysis_10072026' / 'gmaps_data' / 'master_corpus_final.parquet'
DIMS   = {'approach':    ROOT / 'data_annotation/stage_3_subs/approach',
          'context':     ROOT / 'data_annotation/stage_3_subs/context',
          'connotation': ROOT / 'data_annotation/stage_3_subs/connotation',
          'theme':       ROOT / 'data_annotation/stage_3_subs/themes'}
CLASSIFIER = {'approach': 'approach_classifier.py', 'context': 'context_classifier.py',
              'connotation': 'connotation_classifier.py', 'theme': 'themes_classifier.py'}
SEED = 20260710
N_RELABEL, N_BACKTRANS, MIN_PER_STRATUM = 300, 150, 3

def label_set(v):
    """canonical list/array -> sorted lowercase tuple"""
    if v is None or (isinstance(v, float) and pd.isna(v)): return tuple()
    return tuple(sorted(str(x).strip().lower() for x in list(v)))

canon = pd.read_parquet(CANON)
canon['review_id'] = canon['review_id'].astype(str)
for d in DIMS:
    canon[f'{d}_orig'] = canon[d].map(label_set)
canon['stratum'] = (canon['approach_orig'].map('|'.join) + ' // ' +
                    canon['connotation_orig'].map('|'.join) + ' // ' +
                    canon['context_orig'].map('|'.join))

sample_path = RC / 'samples' / 'relabel_sample_300.parquet'
if sample_path.exists():
    sample = pd.read_parquet(sample_path)
    print(f'loaded existing sample ({len(sample)} rows) — delete the file to regenerate')
else:
    sizes = canon['stratum'].value_counts()
    alloc = (sizes / sizes.sum() * N_RELABEL).round().astype(int)
    alloc = alloc.clip(lower=MIN_PER_STRATUM).clip(upper=sizes)   # floor, capped by stratum size
    parts = [canon[canon['stratum'] == s].sample(n=int(n), random_state=SEED)
             for s, n in alloc.items()]
    sample = pd.concat(parts, ignore_index=True)
    keep = ['review_id', 'org_review', 'trans_review', 'stratum'] + [f'{d}_orig' for d in DIMS]
    sample = sample[keep].copy()
    for d in DIMS:                                   # tuples -> JSON strings for parquet stability
        sample[f'{d}_orig'] = sample[f'{d}_orig'].map(list)
    sample.to_parquet(sample_path, index=False)
    print(f'sample written: {len(sample)} rows across {sample.stratum.nunique()} strata')

strata_table = (sample.groupby('stratum').size().rename('sampled')
                .to_frame().join(canon['stratum'].value_counts().rename('population')))
display(strata_table.sort_values('population', ascending=False).head(15))
print(f'... {len(strata_table)} strata total | sampled {len(sample)} of {len(canon)}')

## 2. Experiment A — rerun the classifiers on the sample

Each dimension classifier is invoked through its own **production CLI script** (not re-implemented
here) so prompts, parsing, retry logic and model settings are guaranteed identical. Each run writes to
its own directory under `runs/relabel_<dim>/`; the cell skips any run whose output already exists.

`--batch-size 1 --chunk-size 10` matches the production reclassification settings.

In [ ]:
def run_classifier(dim, input_parquet, run_dir):
    """Invoke a dimension classifier CLI.

    Idempotent: if every review in the input already appears in the run's
    processed_success.csv the subprocess is skipped; otherwise the classifier is
    (re)run — its own resumption logic classifies only the missing reviews.
    """
    out_csv = run_dir / 'processed_success.csv'
    want = set(pd.read_parquet(input_parquet, columns=['review_id'])['review_id'].astype(str))
    if out_csv.exists():
        have = set(pd.read_csv(out_csv)['review_id'].astype(str))
        if want <= have:
            print(f'[{dim}] already done ({len(want)} of {len(have)} rows present) — skipping')
            return
        print(f'[{dim}] {len(want - have)} reviews missing — resuming')
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, CLASSIFIER[dim], 'classify',
           '--input', str(input_parquet), '--base', str(run_dir),
           '--output-dir', str(run_dir), '--chunk-size', '10', '--batch-size', '1']
    print(f'[{dim}] running …')
    res = subprocess.run(cmd, cwd=DIMS[dim], capture_output=True, text=True)
    tail = '\n'.join((res.stdout + res.stderr).splitlines()[-6:])
    print(tail)
    assert out_csv.exists(), f'{dim} relabel run produced no output — inspect logs in {run_dir}'

for dim in DIMS:
    run_classifier(dim, sample_path, RC / 'runs' / f'relabel_{dim}')

## 3. Experiment B — back-translation (English → German → English)

We take a stratified 150-review subset of the Experiment A sample (so every back-translated review has
a paired plain rerun). Both translation directions use the same LLM at temperature 0 with a minimal
instruction, and results are **cached** to `samples/backtranslation_cache.parquet` — re-running this
cell costs nothing once the cache is complete.

The classifier scripts read the review text from the `trans_review` column, so the back-translated
file simply puts the back-translation into `trans_review` (originals are kept in
`trans_review_original` / `trans_review_german` for inspection).

In [ ]:
import os, time
from concurrent.futures import ThreadPoolExecutor
import requests
from dotenv import load_dotenv

load_dotenv(ROOT / 'data_annotation/stage_3_subs/all_subs/.env')
KEYS  = [k.strip() for k in os.getenv('OPENROUTER_API_KEYS', '').split(',') if k.strip()]
MODEL = 'anthropic/claude-sonnet-4-6'

def translate(text, target_lang, key_offset):
    """Translate via the LLM. Rotates through ALL api keys across attempts so a
    credit-exhausted key cannot doom a review (keys hitting 402 stay exhausted)."""
    msg = [{'role': 'system',
            'content': f'You are a professional translator. Translate the user text into {target_lang}. '
                       'Return ONLY the translation, no commentary.'},
           {'role': 'user', 'content': text}]
    n_attempts = max(4, len(KEYS))
    for attempt in range(n_attempts):
        key = KEYS[(key_offset + attempt) % len(KEYS)]
        try:
            r = requests.post('https://openrouter.ai/api/v1/chat/completions',
                              headers={'Authorization': f'Bearer {key}'},
                              json={'model': MODEL, 'messages': msg, 'temperature': 0.0,
                                    'max_tokens': 2000},
                              timeout=90)
            if r.status_code == 200:
                out = r.json()['choices'][0]['message']['content'].strip()
                if out:
                    return out
            elif r.status_code in (402, 401):
                continue                    # bad/exhausted key: try the next one immediately
        except requests.RequestException:
            pass
        time.sleep(1.5 * (attempt + 1))
    return None

bt_sample_path = RC / 'samples' / 'backtrans_sample_150.parquet'
cache_path     = RC / 'samples' / 'backtranslation_cache.parquet'

if bt_sample_path.exists():
    bt = pd.read_parquet(bt_sample_path)
    print(f'loaded existing back-translated sample ({len(bt)} rows)')
else:
    # stratified half of the relabel sample
    rng_alloc = (sample.groupby('stratum').size() / len(sample) * N_BACKTRANS).round().astype(int).clip(lower=1)
    subset = pd.concat([g.sample(n=min(len(g), int(rng_alloc[s])), random_state=SEED + 1)
                        for s, g in sample.groupby('stratum')], ignore_index=True)

    cache = pd.read_parquet(cache_path) if cache_path.exists() else pd.DataFrame(
        columns=['review_id', 'german', 'back_english'])
    cache = cache.set_index('review_id')

    todo = [r for r in subset.itertuples()
            if r.review_id not in cache.index or pd.isna(cache.loc[r.review_id, 'back_english'])]
    print(f'{len(subset)} in subset, {len(todo)} need translating (rest cached)')

    def work(row, i):
        de = translate(row.trans_review, 'German', i)
        en = translate(de, 'English', i + 1) if de else None
        return row.review_id, de, en

    if todo:
        with ThreadPoolExecutor(max_workers=8) as ex:
            for rid, de, en in ex.map(lambda t: work(*t), [(r, i) for i, r in enumerate(todo)]):
                cache.loc[rid, ['german', 'back_english']] = [de, en]
        cache.reset_index().to_parquet(cache_path, index=False)

    bt = subset.merge(cache.reset_index(), on='review_id', validate='one_to_one')
    failed = bt['back_english'].isna().sum()
    if failed:
        print(f'⚠ {failed} reviews failed translation and are excluded')
        bt = bt[bt['back_english'].notna()]
    bt = bt.rename(columns={'trans_review': 'trans_review_original', 'german': 'trans_review_german'})
    bt['trans_review'] = bt['back_english']
    bt.to_parquet(bt_sample_path, index=False)
    print(f'back-translated sample written: {len(bt)} rows')

ex_row = bt.iloc[0]
print('\n--- example ---')
print('ORIGINAL EN :', str(ex_row['trans_review_original'])[:220])
print('GERMAN      :', str(ex_row['trans_review_german'])[:220])
print('BACK TO EN  :', str(ex_row['trans_review'])[:220])

In [ ]:
for dim in DIMS:
    run_classifier(dim, bt_sample_path, RC / 'runs' / f'backtrans_{dim}')

## 4. Comparing labels

For every review we now have up to three label sets per dimension: **original** (canonical),
**relabel** (Experiment A), **back-translation** (Experiment B). Metrics:

- **exact-set stability** — share of reviews whose new label set is identical to the original;
- **mean Jaccard** — partial-credit overlap, separates "total flips" from "one label added/dropped";
- **label flow table** — for unstable reviews, which original set became which new set (reveals
  *systematic* drift, e.g. `coded → implicit`, vs symmetric noise);
- **paired translation effect** — on the 150 paired reviews: how often is the back-translated label
  wrong *while the plain rerun of the same review is right*? The excess of `only-backtrans-unstable`
  over `only-relabel-unstable` (with an exact binomial test on the discordant pairs, McNemar's logic)
  is the translation-specific effect.

In [ ]:
def load_run(dim, run_dir):
    df = pd.read_csv(run_dir / 'processed_success.csv')
    df['review_id'] = df['review_id'].astype(str)
    col = 'theme' if dim == 'theme' else dim
    def parse(v):
        s = str(v).strip()
        toks = re.findall(r"'([^']+)'|\"([^\"]+)\"", s) if s.startswith('[') else []
        labels = [a or b for a, b in toks] if toks else [p for p in re.split(r'[,/]', s.strip('[]')) if p.strip()]
        return tuple(sorted(t.strip().lower() for t in labels if t.strip()))
    return df.set_index('review_id')[col].map(parse)

orig = {d: pd.Series([tuple(v) for v in sample[f'{d}_orig']],
                     index=sample['review_id'].astype(str)) for d in DIMS}

rows, flows = [], {}
for exp, base_df, n_label in (('relabel', sample, 'relabel'), ('backtrans', bt, 'backtrans')):
    ids = base_df['review_id'].astype(str)
    for dim in DIMS:
        new = load_run(dim, RC / 'runs' / f'{exp}_{dim}')
        o = orig[dim].reindex(ids); n = new.reindex(ids)
        ok = o.notna() & n.notna()
        o, n = o[ok], n[ok]
        exact = (o == n).mean()
        jac = np.mean([len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(o, n)])
        rows.append({'experiment': exp, 'dimension': dim, 'n': int(ok.sum()),
                     'exact_stability': exact, 'mean_jaccard': jac})
        fl = pd.DataFrame({'orig': o.map('|'.join), 'new': n.map('|'.join)})
        flows[(exp, dim)] = (fl[fl.orig != fl.new].groupby(['orig', 'new']).size()
                             .sort_values(ascending=False))

stability = pd.DataFrame(rows).pivot(index='dimension', columns='experiment',
                                     values=['exact_stability', 'mean_jaccard', 'n'])
display(stability.round(3))
stability.to_csv(RC / 'results' / 'stability_summary.csv')

print('\nLabel flows (unstable reviews only), relabel experiment:')
for dim in DIMS:
    f = flows[('relabel', dim)]
    if len(f):
        print(f'\n--- {dim} ---'); print(f.head(8).to_string())

In [ ]:
from scipy.stats import binomtest

print('Paired analysis on the back-translated reviews: is instability HIGHER under back-translation')
print('than under a plain rerun of the very same review? (McNemar logic on discordant pairs)\n')
paired_rows = []
ids150 = bt['review_id'].astype(str)
for dim in DIMS:
    re_new = load_run(dim, RC / 'runs' / f'relabel_{dim}').reindex(ids150)
    bt_new = load_run(dim, RC / 'runs' / f'backtrans_{dim}').reindex(ids150)
    o = orig[dim].reindex(ids150)
    ok = o.notna() & re_new.notna() & bt_new.notna()
    A = (o[ok] == re_new[ok])          # rerun stable
    B = (o[ok] == bt_new[ok])          # backtrans stable
    only_bt_unstable = int((A & ~B).sum())
    only_re_unstable = int((~A & B).sum())
    disc = only_bt_unstable + only_re_unstable
    p = binomtest(only_bt_unstable, disc, 0.5).pvalue if disc else np.nan
    paired_rows.append({'dimension': dim, 'n_pairs': int(ok.sum()),
                        'rerun_stability': A.mean(), 'backtrans_stability': B.mean(),
                        'only_backtrans_unstable': only_bt_unstable,
                        'only_rerun_unstable': only_re_unstable, 'mcnemar_p': p})
paired = pd.DataFrame(paired_rows)
display(paired.round(3))
paired.to_csv(RC / 'results' / 'translation_effect_paired.csv', index=False)

unstable = []
for dim in DIMS:
    bt_new = load_run(dim, RC / 'runs' / f'backtrans_{dim}').reindex(ids150)
    o = orig[dim].reindex(ids150)
    m = o.notna() & bt_new.notna() & (o != bt_new)
    for rid in o[m].index:
        unstable.append({'review_id': rid, 'dimension': dim,
                         'original': '|'.join(o[rid]), 'backtrans': '|'.join(bt_new[rid])})
pd.DataFrame(unstable).to_csv(RC / 'results' / 'backtrans_label_changes.csv', index=False)
print(f"\n{len(unstable)} dimension-level label changes under back-translation exported for qualitative review")

## 5. Shared helper — back-translation subset for the two gate classifiers

Experiments C and D below reuse the *same* EN→DE→EN translation cache as Experiment B
(`samples/backtranslation_cache.parquet`, keyed by `review_id`) rather than building a separate one —
a cached review's back-translation doesn't depend on which classifier is being tested, so this adds
no extra translation cost for any review that Experiment B (or C, or D) has already translated.

In [ ]:
def make_backtrans_subset(sample_df, stratify_col, id_col='review_id', text_col='trans_review',
                           n=N_BACKTRANS, seed=SEED):
    """Stratified back-translation subset, sharing Experiment B's translate() function and
    cache_path — same idempotent cache-then-translate-the-rest pattern as Experiment B's cell."""
    alloc = (sample_df.groupby(stratify_col).size() / len(sample_df) * n).round().astype(int).clip(lower=1)
    subset = pd.concat([g.sample(n=min(len(g), int(alloc[s])), random_state=seed)
                        for s, g in sample_df.groupby(stratify_col)], ignore_index=True)

    cache = pd.read_parquet(cache_path) if cache_path.exists() else pd.DataFrame(
        columns=['review_id', 'german', 'back_english'])
    cache = cache.set_index('review_id')

    todo = [r for r in subset.itertuples()
            if getattr(r, id_col) not in cache.index or pd.isna(cache.loc[getattr(r, id_col), 'back_english'])]
    print(f'{len(subset)} in subset, {len(todo)} need translating (rest cached)')

    def work(row, i):
        text = getattr(row, text_col)
        de = translate(text, 'German', i)
        en = translate(de, 'English', i + 1) if de else None
        return getattr(row, id_col), de, en

    if todo:
        with ThreadPoolExecutor(max_workers=8) as ex:
            for rid, de, en in ex.map(lambda t: work(*t), [(r, i) for i, r in enumerate(todo)]):
                cache.loc[rid, ['german', 'back_english']] = [de, en]
        cache.reset_index().to_parquet(cache_path, index=False)

    bt = subset.merge(cache.reset_index(), on=id_col, validate='one_to_one')
    failed = bt['back_english'].isna().sum()
    if failed:
        print(f'⚠ {failed} reviews failed translation and are excluded')
        bt = bt[bt['back_english'].notna()]
    bt = bt.rename(columns={text_col: f'{text_col}_original', 'german': f'{text_col}_german'})
    bt[text_col] = bt['back_english']
    return bt

# Only stage_1_social and stage_2_racial_reclassed are tested here — stage_2_racial (the earlier
# broad screen stage_2_racial_reclassed superseded) is out of scope for this robustness check.
BIN_DIR    = {'social':         ROOT / 'data_annotation/stage_1_social',
              'racial_reclass': ROOT / 'data_annotation/stage_2_racial_reclassed'}
BIN_SCRIPT = {'social':         'social_classifier.py',
              'racial_reclass': 'racial_classifier_parallelization_examples.py'}
BIN_BATCH  = {'social': 5, 'racial_reclass': 1}   # matches each script's own logged production batch_size
BIN_COL    = {'social': 'social', 'racial_reclass': 'racialised'}

def run_binary_classifier(key, input_parquet, run_dir, chunk_size=50):
    """Same idempotent-subprocess pattern as run_classifier() in section 2, generalised to the
    binary social/racial gate classifiers (different CLI script, single yes/no output column)."""
    out_csv = run_dir / 'processed_success.csv'
    want = set(pd.read_parquet(input_parquet, columns=['review_id'])['review_id'].astype(str))
    if out_csv.exists():
        have = set(pd.read_csv(out_csv)['review_id'].astype(str))
        if want <= have:
            print(f'[{key}] already done ({len(want)} of {len(have)} rows present) — skipping')
            return
        print(f'[{key}] {len(want - have)} reviews missing — resuming')
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, BIN_SCRIPT[key], 'classify',
           '--input', str(input_parquet), '--base', str(run_dir),
           '--output-dir', str(run_dir), '--chunk-size', str(chunk_size),
           '--batch-size', str(BIN_BATCH[key])]
    print(f'[{key}] running …')
    res = subprocess.run(cmd, cwd=BIN_DIR[key], capture_output=True, text=True)
    tail = '\n'.join((res.stdout + res.stderr).splitlines()[-6:])
    print(tail)
    assert out_csv.exists(), f'{key} run produced no output — inspect logs in {run_dir}'

def load_binary(key, run_dir):
    df = pd.read_csv(run_dir / 'processed_success.csv')
    df['review_id'] = df['review_id'].astype(str)
    return df.set_index('review_id')[BIN_COL[key]].astype(str).str.strip().str.lower()

print('helpers ready')

## 6. Experiment C — stage 1 social-classifier stability

`stage_1_social` is the very first gate in the whole pipeline: applied to all 1,265,806 reviews in the
corpus, it decides which 504,072 even get considered for racialisation at all. Errors here are the
highest-leverage kind — a review misclassified as not-social never reaches any downstream check,
including everything above. Same two-part design as Experiments A/B, now against the full-corpus
scope instead of the racialised-review scope.

| design decision | value | rationale |
|---|---|---|
| population | 1,265,806 reviews (`master_corpus_final.parquet`) | the classifier's actual input scope |
| sample | n=300, stratified by canonical `social` label | proportional allocation is fine here — the split is ~40/60 (504,072 yes / 761,733 no), not a rare-class problem like Experiment D below |
| back-translation subset | n=150, stratified subset of the 300 | paired with a same-review rerun, as in Experiment B |
| classifier | `social_classifier.py`, batch_size=5 | matches the production run's logged `batch_size=5` |

Estimated added cost: ~300 classify calls (batch_size 5 → ~60 API calls) + 150 back-translation calls ≈ **$2–3**.

In [ ]:
social_sample_path = RC / 'samples' / 'social_sample_300.parquet'
if social_sample_path.exists():
    social_sample = pd.read_parquet(social_sample_path)
    print(f'loaded existing sample ({len(social_sample)} rows) — delete the file to regenerate')
else:
    master_social = pd.read_parquet(MASTER, columns=['review_id', 'org_review', 'trans_review', 'social'])
    master_social = master_social[master_social['social'].isin(['yes', 'no'])].copy()
    master_social['review_id'] = master_social['review_id'].astype(str)
    sizes = master_social['social'].value_counts()
    alloc = (sizes / sizes.sum() * N_RELABEL).round().astype(int).clip(lower=MIN_PER_STRATUM).clip(upper=sizes)
    parts = [master_social[master_social['social'] == s].sample(n=int(n), random_state=SEED + 10)
             for s, n in alloc.items()]
    social_sample = (pd.concat(parts, ignore_index=True)
                        .rename(columns={'social': 'social_orig'}))
    social_sample.to_parquet(social_sample_path, index=False)
    print(f'sample written: {len(social_sample)} rows')

print(social_sample['social_orig'].value_counts())

run_binary_classifier('social', social_sample_path, RC / 'runs' / 'relabel_social')

In [ ]:
social_bt_path = RC / 'samples' / 'social_backtrans_sample_150.parquet'
if social_bt_path.exists():
    social_bt = pd.read_parquet(social_bt_path)
    print(f'loaded existing back-translated sample ({len(social_bt)} rows)')
else:
    social_bt = make_backtrans_subset(social_sample, stratify_col='social_orig')
    social_bt.to_parquet(social_bt_path, index=False)
    print(f'back-translated sample written: {len(social_bt)} rows')

run_binary_classifier('social', social_bt_path, RC / 'runs' / 'backtrans_social')

In [ ]:
orig_social = social_sample.set_index('review_id')['social_orig'].astype(str).str.strip().str.lower()
re_social = load_binary('social', RC / 'runs' / 'relabel_social').reindex(orig_social.index)
exact_relabel = (orig_social == re_social).mean()
print(f"social relabel exact-match stability: {exact_relabel:.3f}  (n={orig_social.notna().sum()})")

bt_ids = social_bt['review_id'].astype(str)
orig_bt = orig_social.reindex(bt_ids)
bt_social = load_binary('social', RC / 'runs' / 'backtrans_social').reindex(bt_ids)
exact_bt = (orig_bt == bt_social).mean()
print(f"social back-translation exact-match stability: {exact_bt:.3f}  (n={len(bt_ids)})")

re_paired = re_social.reindex(bt_ids)
A = (orig_bt == re_paired)     # rerun stable
B = (orig_bt == bt_social)     # backtrans stable
only_bt_unstable = int((A & ~B).sum())
only_re_unstable = int((~A & B).sum())
disc = only_bt_unstable + only_re_unstable
p = binomtest(only_bt_unstable, disc, 0.5).pvalue if disc else np.nan

social_summary = pd.DataFrame([{
    'dimension': 'social', 'n_pairs': int(len(bt_ids)),
    'rerun_stability': A.mean(), 'backtrans_stability': B.mean(),
    'only_backtrans_unstable': only_bt_unstable, 'only_rerun_unstable': only_re_unstable, 'mcnemar_p': p
}])
print('\nConfusion matrix (canonical social label vs. rerun label):')
print(pd.crosstab(orig_social.rename('canonical'), re_social.rename('rerun')))
display(social_summary.round(3))

## 7. Experiment D — stage 2 racial-classifier (reclassed) stability

Per project convention, `stage_2_racial` (the earlier, broader-triggering screen) is **not** the
classifier of record — `stage_2_racial_reclassed` is, and this experiment tests only that script.
It is applied directly to a stratified sample of the social-yes population and its output compared
to the canonical `racialised` label, using the same rerun + back-translation design as every other
experiment in this notebook.

| design decision | value | rationale |
|---|---|---|
| population | 504,072 social-yes reviews | `stage_2_racial_reclassed`'s scope for this check |
| sample | n=300, **50/50 case-control** (150 canonical-yes + 150 canonical-no) | `racialised`=yes is only ~1.5% of this population (7,768/504,072) — a proportional n=300 sample would carry ~5 yes-rows, far too few to say anything about false negatives |
| back-translation subset | n=150, stratified subset of the 300 | paired with a same-review rerun |
| classifier | `racial_classifier_parallelization_examples.py`, batch_size=1 | matches the script's own logged production setting |

Estimated added cost: ~300 classify calls + 150 back-translation pairs, run twice (relabel + backtrans) ≈ **$3–5**.

In [ ]:
racial_sample_path = RC / 'samples' / 'racial_sample_300.parquet'
if racial_sample_path.exists():
    racial_sample = pd.read_parquet(racial_sample_path)
    print(f'loaded existing sample ({len(racial_sample)} rows) — delete the file to regenerate')
else:
    master_racial = pd.read_parquet(MASTER, columns=['review_id', 'org_review', 'trans_review', 'social', 'racialised'])
    social_yes = master_racial[master_racial['social'] == 'yes'].copy()
    social_yes['review_id'] = social_yes['review_id'].astype(str)
    N_YES, N_NO = 150, 150
    yes_pool = social_yes[social_yes['racialised'] == 'yes']
    no_pool  = social_yes[social_yes['racialised'] == 'no']
    racial_sample = pd.concat([
        yes_pool.sample(n=min(N_YES, len(yes_pool)), random_state=SEED + 20),
        no_pool.sample(n=min(N_NO, len(no_pool)), random_state=SEED + 21),
    ], ignore_index=True).rename(columns={'racialised': 'racialised_orig'}).drop(columns=['social'])
    racial_sample.to_parquet(racial_sample_path, index=False)
    print(f'sample written: {len(racial_sample)} rows')

print(racial_sample['racialised_orig'].value_counts())

run_binary_classifier('racial_reclass', racial_sample_path, RC / 'runs' / 'relabel_racial_reclass')

In [ ]:
racial_bt_path = RC / 'samples' / 'racial_backtrans_sample_150.parquet'
if racial_bt_path.exists():
    racial_bt = pd.read_parquet(racial_bt_path)
    print(f'loaded existing back-translated sample ({len(racial_bt)} rows)')
else:
    racial_bt = make_backtrans_subset(racial_sample, stratify_col='racialised_orig')
    racial_bt.to_parquet(racial_bt_path, index=False)
    print(f'back-translated sample written: {len(racial_bt)} rows')

run_binary_classifier('racial_reclass', racial_bt_path, RC / 'runs' / 'backtrans_racial_reclass')

In [ ]:
orig_racial = racial_sample.set_index('review_id')['racialised_orig'].astype(str).str.strip().str.lower()
re_racial = load_binary('racial_reclass', RC / 'runs' / 'relabel_racial_reclass').reindex(orig_racial.index)
exact_relabel_r = (orig_racial == re_racial).mean()
print(f"racialised (reclassed) relabel exact-match stability: {exact_relabel_r:.3f}  (n={orig_racial.notna().sum()})")

bt_ids_r = racial_bt['review_id'].astype(str)
orig_bt_r = orig_racial.reindex(bt_ids_r)
bt_racial = load_binary('racial_reclass', RC / 'runs' / 'backtrans_racial_reclass').reindex(bt_ids_r)
exact_bt_r = (orig_bt_r == bt_racial).mean()
print(f"racialised (reclassed) back-translation exact-match stability: {exact_bt_r:.3f}  (n={len(bt_ids_r)})")

re_paired_r = re_racial.reindex(bt_ids_r)
A_r = (orig_bt_r == re_paired_r)
B_r = (orig_bt_r == bt_racial)
only_bt_unstable_r = int((A_r & ~B_r).sum())
only_re_unstable_r = int((~A_r & B_r).sum())
disc_r = only_bt_unstable_r + only_re_unstable_r
p_r = binomtest(only_bt_unstable_r, disc_r, 0.5).pvalue if disc_r else np.nan

racial_summary = pd.DataFrame([{
    'dimension': 'racialised', 'n_pairs': int(len(bt_ids_r)),
    'rerun_stability': A_r.mean(), 'backtrans_stability': B_r.mean(),
    'only_backtrans_unstable': only_bt_unstable_r, 'only_rerun_unstable': only_re_unstable_r, 'mcnemar_p': p_r
}])
print('\nConfusion matrix (canonical racialised label vs. rerun label):')
print(pd.crosstab(orig_racial.rename('canonical'), re_racial.rename('rerun')))
display(racial_summary.round(3))

## 8. Combined export — all four robustness checks together

In [ ]:
combined_paired = pd.concat([paired, social_summary, racial_summary], ignore_index=True)
display(combined_paired.round(3))
combined_paired.to_csv(RC / 'results' / 'translation_effect_paired_full.csv', index=False)

gate_stability = pd.DataFrame([
    {'gate': 'social', 'n_relabel': int(orig_social.notna().sum()), 'relabel_exact_stability': exact_relabel,
     'n_backtrans': int(len(bt_ids)), 'backtrans_exact_stability': exact_bt},
    {'gate': 'racialised', 'n_relabel': int(orig_racial.notna().sum()), 'relabel_exact_stability': exact_relabel_r,
     'n_backtrans': int(len(bt_ids_r)), 'backtrans_exact_stability': exact_bt_r},
])
display(gate_stability.round(3))
gate_stability.to_csv(RC / 'results' / 'gate_stability_summary.csv', index=False)

social_changes = pd.DataFrame([
    {'review_id': rid, 'gate': 'social', 'original': orig_bt[rid], 'backtrans': bt_social[rid]}
    for rid in orig_bt.index if pd.notna(orig_bt[rid]) and pd.notna(bt_social[rid]) and orig_bt[rid] != bt_social[rid]
])
racial_changes = pd.DataFrame([
    {'review_id': rid, 'gate': 'racialised', 'original': orig_bt_r[rid], 'backtrans': bt_racial[rid]}
    for rid in orig_bt_r.index if pd.notna(orig_bt_r[rid]) and pd.notna(bt_racial[rid]) and orig_bt_r[rid] != bt_racial[rid]
])
pd.concat([social_changes, racial_changes], ignore_index=True).to_csv(
    RC / 'results' / 'gate_backtrans_label_changes.csv', index=False)
print(f"{len(social_changes)} social + {len(racial_changes)} racialised label changes under back-translation exported")
print("\nOutputs: results/translation_effect_paired_full.csv (all 6 checks), "
      "results/gate_stability_summary.csv, results/gate_backtrans_label_changes.csv")

## 9. Reading the results

- **`exact_stability` (relabel)** is the noise floor. Anything ≥ ~0.90 means the classifier is highly
  deterministic in practice; label flows show whether residual noise is concentrated in particular
  categories (those categories' *analysis* results should be treated with wider error bars).
- **`backtrans_stability` vs `rerun_stability`** (paired table): if the McNemar p-value is small and
  `only_backtrans_unstable` clearly exceeds `only_rerun_unstable`, labels are sensitive to
  translation phrasing — quantify the size and, if material, consider (a) reporting it as a robustness
  bound in the paper, or (b) classifying a bilingual subsample on original German to triangulate.
- `results/backtrans_label_changes.csv` lists every flipped review for qualitative inspection — read a
  dozen: are flips genuine ambiguity (both labels defensible) or translation artifacts (meaning
  changed)?
- **Experiments C and D (sections 6–7) extend the same logic to the two binary gate classifiers**
  (`social`, and `racialised` via `stage_2_racial_reclassed` — the classifier of record;
  `stage_2_racial` itself is out of scope) that Experiments A/B don't touch — they assume those
  upstream labels are already correct and only probe the downstream sub-dimensions. Gates matter more
  per-error than sub-dimensions: a flipped `social` or `racialised` label doesn't just mislabel one
  dimension, it adds or removes the review from the whole analysis population.
  `results/gate_stability_summary.csv` and the `racialised` row of
  `results/translation_effect_paired_full.csv` are the numbers that matter most for the paper's
  overall robustness claim, precisely because they sit upstream of everything else here.

Outputs: `results/stability_summary.csv`, `results/translation_effect_paired.csv`,
`results/backtrans_label_changes.csv` (Experiments A/B); `results/gate_stability_summary.csv`,
`results/gate_backtrans_label_changes.csv`, `results/translation_effect_paired_full.csv` (all six
checks combined, Experiments A–D).